In [1]:
import numpy as np  # import numerical python
import matplotlib.pyplot as plt  # import plotting functions
import seaborn as sns  # import nicer plotting functions
import polars as pl  # import polars to import data
import tifffile as tiff
from tifffile import imwrite, imread
from copy import deepcopy
import os
import time

import sys
sys.path.append("..")

from src import IOFunctions

IO = IOFunctions.IO_Functions()

from src import Multicolour_Simulation_Functions

MSF = Multicolour_Simulation_Functions.MultiC_Sim_Funcs()

from src import PlottingFunctions

plotter = PlottingFunctions.Plotter()

from src import ImageAnalysisFunctions

I_AF = ImageAnalysisFunctions.Image_Analysis_Functions()

from src import sCMOSFunctions

sCMOS = sCMOSFunctions.sCMOS_Functions()

from src import PSFFunctions

PSF = PSFFunctions.PSF_Functions()

from src import SpectralFunctions

S_F = SpectralFunctions.Spectral_Funcs()

from src import MaskFunctions

M_F = MaskFunctions.Mask_Functions()

from src import SpotDetectionFunctions

SD_F = SpotDetectionFunctions.SpotDetection_Functions()

from src import SR_Functions

SRes_F = SR_Functions.SuperRes_Functions()

from src import HelperFunctions

H_F = HelperFunctions.Helper_Functions()

In [2]:
#data_folder = '/home/jbeckwith/Documents/Cambridge University Dropbox/Joseph Beckwith/Chemistry/Lee/Data/Salix/CS505CU_Calibration/old/'
data_folder = '/home/jbeckwith/Documents/Cambridge University Dropbox/Joseph Beckwith/Chemistry/Lee/Data/Salix/Ximea_Calibration/'
#data_folder = r'C:\Users\jsb92\Cambridge University Dropbox\Joseph Beckwith\Chemistry\Lee\Data\Salix\Ximea_Calibration'
gain = IO.read_tiff(os.path.join(data_folder, "gain.tif"))
offset = IO.read_tiff(os.path.join(data_folder, "offset.tif"))
variance = IO.read_tiff(os.path.join(data_folder, "variance.tif"))
readnoise = IO.read_tiff(os.path.join(data_folder, "readnoise.tif"))
rqe = IO.read_tiff(os.path.join(data_folder, "rqe.tif"))
R, G, B, wavelength = S_F.getpixelefficiency()


In [3]:
wavelength = wavelength
pixel_QYs = np.vstack([B, G, R])
image_size = 12
camera_parameters = {}
masks = M_F.get_masks(size_x=image_size, size_y=image_size)
camera_parameters["gain"] = np.full((image_size, image_size), 1.2)
camera_parameters["offset"] = np.full((image_size, image_size), 100)
camera_parameters["variance"] = np.full((image_size, image_size), 0.9)
camera_parameters["readnoise"] = np.full((image_size, image_size), 0.2)
camera_parameters["rqe"] = np.full((image_size, image_size), 1)
camera_parameters["pixel_QYs"] = pixel_QYs
camera_parameters["pixel_order"] = ['B', 'G', 'R']
camera_parameters["pixel_order_indices"] = [0, 1, 2]

camera_parameters["masks"] = M_F.get_masks(size_x = image_size, size_y = image_size)
camera_parameters["masks"] = masks

In [ ]:
dye_data_folders = np.array([r'/media/jbeckwith/Ezra Seagat/JSB/20250507_4Dyes/data/LD655/LD655_20mW655_LP638_SP785_1/',
                             r'/media/jbeckwith/Ezra Seagat/JSB/20250507_4Dyes/data/CF680_reappliedOil/CF680_60mW655_LP638_SP785_1/',
                             r'/media/jbeckwith/Ezra Seagat/JSB/20250507_4Dyes/data/CF500/CF500_40mW_LP488_BP520-44_SP785_1/',
                             r'/media/jbeckwith/Ezra Seagat/JSB/20250507_4Dyes/data/ATTO488/ATTO488_40mW_LP488_BP520-44_SP785_1/',
                            '/media/jbeckwith/Ezra Seagat/JSB/20250325 Ximea camera install/ATTO_565_100pMPVA_10mW_561/Atto565_10mW_561_lp561_bp_582_64_5_Scan_1/',
                            '/media/jbeckwith/Ezra Seagat/JSB/20250325 Ximea camera install/ATTO_647N_100pMPVA_40mW_640/ATTO647N_40mW_640_638LP_Scan_1/',
                            '/media/jbeckwith/Ezra Seagat/JSB/20250325 Ximea camera install/BODIPY_493_503_100pM_PVA/BODIPY_493_503_40mW_488_LP488_BP_520_44_Scan2_2/'])

In [4]:
import types
smoothing_function = types.SimpleNamespace()
smoothing_function.args = {"sigma" :  1.5}
smoothing_function.extent =  1.5
smoothing_function.smoothing_function = sCMOS.gaussian_filter_stack
smoothing_function.data_arg = "image"

In [ ]:
image_files = H_F.file_search(dye_data_folders[6], '.tif', "")

In [ ]:
photolectron_data, smoothed_data, weights = IO.read_tiff_tophotoelectrons(image_files[1], frame=0, smoothing_function=smoothing_function)

In [ ]:
fig, axs = SRes_F.example_spots_singleframe(photolectron_data, pfa=1e-3, peak_wavelength=0.52)
plt.show()

In [ ]:
pfas = np.array([1e-4, 1e-3, 1e-5, 1e-5, 1e-4, 1e-3, 1e-3])
peak_wavelength = np.array([0.647, 0.647, 0.52, 0.52, 0.565, 0.647, 0.52])

In [ ]:
for i, image_folder in enumerate(dye_data_folders):
    SRes_F.fit_SM_data(image_folder, smoothing_function, gain, offset, rqe, readnoise, pfa=pfas[i], peak_wavelength=peak_wavelength[i])

In [ ]:
new_dye_data_folder = np.array([r'/media/jbeckwith/Ezra Seagat/JSB/20250609_dyes/data/100nMLD655_PLL_40mW638_TIRF_bothlasers_405488561638di_638LP_1/'])

In [ ]:
SRes_F.fit_SM_data(new_dye_data_folder[0], smoothing_function, gain, offset, rqe, readnoise, pfa=1e-4, peak_wavelength=0.647)

In [8]:
QD_folders = np.sort(os.listdir('/media/jbeckwith/Ezra Seagat/JSB/20250606_QDs/data/'))
QD_folders = np.sort(QD_folders)
QD_folders = [os.path.join('/media/jbeckwith/Ezra Seagat/JSB/20250606_QDs/data/', x) for x in QD_folders]

In [9]:
QD_folders

['/media/jbeckwith/Ezra Seagat/JSB/20250606_QDs/data/QD530_LP515_BP650200_51520perc_TIRF_1',
 '/media/jbeckwith/Ezra Seagat/JSB/20250606_QDs/data/QD540_LP515_BP650200_51520perc_TIRF_1',
 '/media/jbeckwith/Ezra Seagat/JSB/20250606_QDs/data/QD550_LP515_BP650200_51520perc_TIRF_1',
 '/media/jbeckwith/Ezra Seagat/JSB/20250606_QDs/data/QD560_LP515_BP650200_51520perc_TIRF_1',
 '/media/jbeckwith/Ezra Seagat/JSB/20250606_QDs/data/QD570_LP515_BP650200_51520perc_TIRF_1',
 '/media/jbeckwith/Ezra Seagat/JSB/20250606_QDs/data/QD580_LP515_BP650200_51520perc_TIRF_1',
 '/media/jbeckwith/Ezra Seagat/JSB/20250606_QDs/data/QD590_LP515_BP650200_51520perc_TIRF_1',
 '/media/jbeckwith/Ezra Seagat/JSB/20250606_QDs/data/QD600_LP515_BP650200_51520perc_TIRF_1',
 '/media/jbeckwith/Ezra Seagat/JSB/20250606_QDs/data/QD600_LP515_BP650200_51560perc_TIRF_1',
 '/media/jbeckwith/Ezra Seagat/JSB/20250606_QDs/data/QD610_LP515_BP650200_51560perc_TIRF_1',
 '/media/jbeckwith/Ezra Seagat/JSB/20250606_QDs/data/QD620_LP515_BP650

In [10]:
peak_wavelengths = np.array([0.53, 0.54, 0.55, 0.56, 0.57, 0.58, 0.59, 0.6, 0.6, 0.61, 0.62, 0.62, 0.63, 0.64, 0.65])
for i, folder in enumerate(QD_folders):
    SRes_F.fit_SM_data(folder, smoothing_function, gain, offset, rqe, readnoise, pfa=1e-4, peak_wavelength=peak_wavelengths[i])

LM fitting: 100%|████████████████████████| 1800/1800 [00:07<00:00, 225.86task/s]


In [41]:
from scipy.stats import entropy
from scipy.stats import wasserstein_distance

In [43]:
A = np.random.normal(size=1000)
B = np.random.normal(size=1000, loc=1)

In [44]:
binsA, edgesA = np.histogram(A, np.linspace(-10, 10, 1000), density=True)
binsB, edgesB = np.histogram(B, np.linspace(-10, 10, 1000), density=True)

In [45]:
wasserstein_distance(binsA, binsB)

0.00150000000000002